In [ ]:
import sys
import warnings
from datetime import date
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from dotenv import dotenv_values
from scipy import stats
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL
from sqlalchemy.exc import OperationalError
from statsmodels.stats.diagnostic import acorr_ljungbox

from energy.domain.equs_gold import DateRange
from energy.domain.equs_series import BatteryEquivalents, totals
from energy.pipelines.flows.equs_gold import battery_equivalents, resumen_equ
from energy.pipelines.flows.equs_report import build_report

BASE_DIR = Path.cwd()  # notebooks/
PROJECT_ROOT = BASE_DIR.parent  # notebooks/ -> raíz del repo
sys.dont_write_bytecode = True  # no crear __pycache__ en data_utils/ (alcance: solo leer)
sys.path.insert(0, str(PROJECT_ROOT))

from data_utils.fancy_barplot import fancy_bars  # noqa: E402

REPO_ROOT = Path.cwd().parent

# Real filenames in data/ (the older notebooks still name them differently).
ENERGY_WORKBOOK = REPO_ROOT / "data" / "Base_de_datos.xlsx"
RECTIFIER_WORKBOOK = REPO_ROOT / "data" / "SCADA-Base de datos F&A.xlsx"
RECTIFIER_SHEET = "Dataset 1"

RELAY_HOST, RELAY_PORT = "127.0.0.1", 5433
EQ_RANGE = DateRange(date(2025, 10, 1), date(2026, 9, 30))  # FY26, all the EQ the database holds
ALPHA = 0.05

LOCAL_TZ = "America/Bogota"  # SCADA timestamps are naive local time (UTC-5, no DST)
WH_PER_KWH = 1000
OPERATING_KWH = 50  # below this in an hour, a rectifier counts as off
PHYSICAL_LIMIT_FACTOR = 2  # a jump above 2 x p99 of operating hours is impossible
RECTIFIERS = {"RECTIFICADOR 1": "R1", "RECTIFICADOR 2": "R2", "RECTIFICADOR 3": "R3"}
ROLLING_DAYS = 7
LAGS = range(-2, 3)
CENT = 0.01

SPANISH_MONTHS = {
    "Ene": 1, "Feb": 2, "Mar": 3, "Abr": 4, "May": 5, "Jun": 6,
    "Jul": 7, "Ago": 8, "Sep": 9, "Oct": 10, "Nov": 11, "Dic": 12,
}  # fmt: skip
WEEKDAYS = ["Lun", "Mar", "Mié", "Jue", "Vie", "Sáb", "Dom"]
SUNDAY = 6

# Energy pair -> (sheet of Base_de_datos, its total row, the EQUS processes it is set against)
PAIRS = {
    "F&A": ("F&A", "F&A", ("FORMACION", "DECORADO")),
    "OREC": ("OREC", "OREC", ("OREC",)),
    "ENSAMBLE": ("Ensamble", "Ensamble", ("ENSAMBLE",)),
}

# The SCADA export has no default style; openpyxl warns about it on every read.
warnings.filterwarnings("ignore", message="Workbook contains no default style")
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({"figure.figsize": (12, 4.5), "axes.titleweight": "bold"})
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")
pd.set_option("display.width", 200)

In [ ]:
env = dotenv_values(REPO_ROOT / ".env")
engine = create_engine(
    URL.create(
        "postgresql+psycopg",
        username=env["POSTGRES_USER"],
        password=env["POSTGRES_PASSWORD"],
        host=RELAY_HOST,
        port=RELAY_PORT,
        database=env["POSTGRES_DB"],
    ),
    connect_args={"options": "-c default_transaction_read_only=on"},
)
try:
    with engine.connect() as connection:
        head = connection.execute(text("SELECT version_num FROM alembic_version")).scalar_one()
        read_only = connection.execute(text("SHOW default_transaction_read_only")).scalar_one()
except OperationalError as error:
    raise RuntimeError(
        f"No database on {RELAY_HOST}:{RELAY_PORT}: start the relay (see the cell above)"
    ) from error
print(f"Connected through the relay: Alembic head {head}, read-only = {read_only}")

In [ ]:
def series_frame(series: BatteryEquivalents) -> pd.DataFrame:
    """One row per day (or month): EQ of each battery process and EQUS BATT, as floats."""
    frame = pd.DataFrame(
        [{"period": t.start, **t.by_process, "EQUS BATT": t.total} for t in totals(series)]
    ).set_index("period")
    frame.index = pd.to_datetime(frame.index)
    return frame.astype("float64")


daily_series = battery_equivalents(EQ_RANGE, frequency="day", engine=engine)
monthly_series = battery_equivalents(EQ_RANGE, frequency="month", engine=engine)
eq_daily = series_frame(daily_series)
eq_monthly = series_frame(monthly_series)
days_with_rows = pd.DatetimeIndex(sorted({row.start for row in daily_series.rows}))

In [ ]:
eq_monthly

In [ ]:
eq_daily

In [ ]:
HOJA, POSICION_HOJA = "Dataset 1", 0

RUTA = Path("../data/SCADA-Base de datos Energia RC.xlsx")
raw = pd.read_excel(RUTA, sheet_name=HOJA)
print(f"Libro: {RUTA.name}")
print(f"\nhojas: {pd.ExcelFile(RUTA).sheet_names}")
print(f"\nHoja cargada: '{HOJA}'")
print(f"\nforma: {raw.shape}")
dtypes_crudos = raw.dtypes.astype(str)

In [ ]:
raw

In [ ]:
period = DateRange(start=date(2025, 10, 1), end=date(2026, 9, 30))
report = build_report(engine, period.start, period.end, source="live")

raw_data = report[0] if isinstance(report, tuple) else report
meses_a_procesar = pd.date_range(start=period.start, end=period.end, freq="MS").date

flat_rows_total = []

# 2. Iterar sobre cada mes del periodo
for m in meses_a_procesar:
    # Llamar a la función para el mes actual
    raw_data = resumen_equ(month=m, engine=engine)

    if not raw_data:
        continue

    raw_data = raw_data[0] if isinstance(raw_data, tuple) else raw_data

    # Metadatos globales del mes
    generado_en = raw_data.get("generado_en")
    desde = raw_data.get("rango", {}).get("desde")
    hasta = raw_data.get("rango", {}).get("hasta")
    mes = raw_data.get("rango", {}).get("meses", [str(m)])[0]

    # 3. Extracción de cuentas de Plantas y Procesos
    for planta_item in raw_data.get("plantas", []):
        planta_name = planta_item.get("planta")
        for proceso_item in planta_item.get("procesos", []):
            proceso_name = proceso_item.get("proceso")
            for sub in proceso_item.get("subsecciones", []):
                flat_rows_total.append(
                    {
                        "generado_en": generado_en,
                        "mes": mes,
                        "rango_desde": desde,
                        "rango_hasta": hasta,
                        "tipo": "cuenta",
                        "fila": sub.get("fila"),
                        "nombre": sub.get("cuenta_stats"),
                        "descripcion": sub.get("descripcion"),
                        "unidad": sub.get("unidad"),
                        "fuente": sub.get("fuente"),
                        "proceso": proceso_name,
                        "planta": planta_name,
                        "categoria": None,
                        "cantidad": sub.get("cantidad"),
                        "factor": sub.get("factor"),
                        "equs": sub.get("equs"),
                    }
                )

    # 4. Extracción de informativos
    for inf in raw_data.get("informativos", []):
        flat_rows_total.append(
            {
                "generado_en": generado_en,
                "mes": mes,
                "rango_desde": desde,
                "rango_hasta": hasta,
                "tipo": "informativo",
                "fila": inf.get("fila"),
                "nombre": inf.get("cuenta_stats"),
                "descripcion": inf.get("descripcion"),
                "unidad": inf.get("unidad"),
                "fuente": None,
                "proceso": None,
                "planta": None,
                "categoria": None,
                "cantidad": inf.get("cantidad"),
                "factor": None,
                "equs": None,
            }
        )

    # 5. Extracción de componentes (equs_poly)
    poly_data = raw_data.get("equs_poly", {})
    for poly in poly_data.get("subsecciones", []):
        flat_rows_total.append(
            {
                "generado_en": generado_en,
                "mes": mes,
                "rango_desde": desde,
                "rango_hasta": hasta,
                "tipo": "poly",
                "fila": poly.get("fila"),
                "nombre": poly.get("cuenta"),
                "descripcion": None,
                "unidad": "UN" if poly.get("cuenta") != "NA" else "KG",
                "fuente": None,
                "proceso": None,
                "planta": "Componentes",
                "categoria": poly.get("categoria"),
                "cantidad": poly.get("piezas"),
                "factor": poly.get("factor_equ"),
                "equs": poly.get("equs"),
            }
        )

# Creación del DataFrame final consolidado con todos los meses
df = pd.DataFrame(flat_rows_total)
df

In [ ]:
df["planta"].value_counts()

In [ ]:
df["planta"].isna().sum()

In [ ]:
query = """
SELECT *
FROM gold.v_reporte_qad
-- Opcional: Agrega un filtro si la vista es muy grande (ej. por mes o fecha)
-- WHERE mes >= '2026-01-01'
;
"""

# Leer los datos directamente a un DataFrame de pandas
df_reporte = pd.read_sql(query, con=engine)

# Vista previa de los primeros registros
df_reporte

In [ ]:
# 'engine' es tu motor de SQLAlchemy previamente configurado
query_equs_diario = """
WITH transacciones_diarias AS (
    SELECT fecha_efectiva, mes, version, cuenta_stats, en_vigor, valor
    FROM gold.v_reporte_qad_regla
    UNION ALL
    SELECT fecha_efectiva, mes, version, cuenta_stats, en_vigor, valor
    FROM gold.v_unit_ship_regla
),
cantidad_diaria AS (
    SELECT
        fecha_efectiva,
        mes,
        version,
        cuenta_stats,
        COALESCE(SUM(valor), 0) AS cantidad_diaria
    FROM transacciones_diarias
    WHERE en_vigor = TRUE
    GROUP BY fecha_efectiva, mes, version, cuenta_stats
)
SELECT
    c.fecha_efectiva,
    c.mes,
    c.version,
    f.proceso,
    c.cuenta_stats,
    f.descripcion,
    c.cantidad_diaria AS cantidad,
    f.factor,
    (c.cantidad_diaria * f.factor) AS equs,
    f.suma_eq
FROM cantidad_diaria c
JOIN silver.factores_equ f
  ON f.version = c.version AND f.cuenta_stats = c.cuenta_stats
WHERE c.fecha_efectiva >= '2020-09-01'  -- Puedes ajustar el rango de fechas
ORDER BY c.fecha_efectiva ASC, c.cuenta_stats ASC;
"""

df_equs_diario = pd.read_sql(query_equs_diario, con=engine)
df_equs_diario

In [ ]:
df_equs_diario["proceso"].unique()

In [ ]:
query_detalle_articulo = """
WITH detalle_transacciones AS (
    SELECT
        fecha_efectiva, mes, version, proceso, cuenta_stats, numero_articulo,
        linea_de_productos, campo_equs_eq, cantidad, valor
    FROM gold.v_reporte_qad_regla
    WHERE en_vigor = TRUE

    UNION ALL

    SELECT
        fecha_efectiva, mes, version, proceso, cuenta_stats, numero_articulo,
        linea_de_productos, campo_equs_eq, cantidad, valor
    FROM gold.v_unit_ship_regla
    WHERE en_vigor = TRUE
)
SELECT
    d.fecha_efectiva,
    d.mes,
    d.version,
    d.proceso,
    d.cuenta_stats,
    f.descripcion AS descripcion_cuenta,
    d.numero_articulo,
    d.linea_de_productos,
    d.campo_equs_eq,
    SUM(d.cantidad) AS cantidad_unidades,
    SUM(d.valor) AS base_calculo_total,
    f.factor,
    SUM(d.valor * f.factor) AS equs_totales,
    f.suma_eq
FROM detalle_transacciones d
JOIN silver.factores_equ f
  ON f.version = d.version
 AND f.cuenta_stats = d.cuenta_stats
GROUP BY
    d.fecha_efectiva,
    d.mes,
    d.version,
    d.proceso,
    d.cuenta_stats,
    f.descripcion,
    d.numero_articulo,
    d.linea_de_productos,
    d.campo_equs_eq,
    f.factor,
    f.suma_eq
ORDER BY
    d.fecha_efectiva ASC,
    d.numero_articulo ASC;
"""

df_detalle_diario = pd.read_sql(query_detalle_articulo, con=engine)
df_detalle_diario

In [ ]:
fancy_bars(df_detalle_diario, x="linea_de_productos")

In [ ]:
query = """
SELECT
    linea_de_productos,
    descripcion
FROM silver.linea_de_productos
WHERE linea_de_productos = '3103';
"""

df_linea = pd.read_sql(query, con=engine)
print(df_linea)

In [ ]:
# Configuración de estilo visual
sns.set_theme(style="whitegrid")
plt.rcParams["font.sans-serif"] = "DejaVu Sans"

# ==============================================================================
# 1. Gráfica: Top 10 Artículos con Mayor Volumen de EQUS Vendidas
# ==============================================================================
query_top_equs_ventas = """
SELECT
    s.numero_articulo,
    COALESCE(i.descripcion, s.numero_articulo) AS nombre_articulo,
    SUM(s.valor * f.factor) AS equs_totales
FROM gold.v_unit_ship_regla s
JOIN silver.factores_equ f
  ON f.version = s.version
 AND f.cuenta_stats = s.cuenta_stats
LEFT JOIN silver.item_master i
       ON i.numero_articulo = s.numero_articulo
WHERE s.en_vigor = TRUE
GROUP BY s.numero_articulo, i.descripcion
ORDER BY equs_totales DESC
LIMIT 10;
"""

df_top_equs = pd.read_sql(query_top_equs_ventas, con=engine)

# Generación del gráfico de barras horizontales
plt.figure(figsize=(12, 6))
ax1 = sns.barplot(data=df_top_equs, x="equs_totales", y="nombre_articulo", palette="viridis")

plt.title("Top 10 Artículos Más Vendidos (EQUS Totales)", fontsize=14, fontweight="bold", pad=15)
plt.xlabel("Baterías Equivalentes (EQUS)", fontsize=11)
plt.ylabel("Artículo", fontsize=11)

# Agregar etiquetas de valor en cada barra
for p in ax1.patches:
    width = p.get_width()
    ax1.annotate(
        f"{width:,.2f}",
        (width, p.get_y() + p.get_height() / 2.0),
        ha="left",
        va="center",
        xytext=(5, 0),
        textcoords="offset points",
        fontsize=10,
    )

plt.tight_layout()
plt.show()


# ==============================================================================
# Evolución Mensual de Producción EQUS por Línea (Código + Descripción)
# ==============================================================================
query_evolucion_mensual_desc = """
SELECT
    r.mes,
    r.linea_de_productos,
    CONCAT(
        r.linea_de_productos,
        ' - ', COALESCE(lp.descripcion,
                        'Sin Descripción'))
        AS linea_con_descripcion,
    SUM(r.valor * f.factor) AS equs_totales
FROM gold.v_reporte_qad_regla r
JOIN silver.factores_equ f
  ON f.version = r.version
 AND f.cuenta_stats = r.cuenta_stats
LEFT JOIN silver.linea_de_productos lp
       ON lp.linea_de_productos = r.linea_de_productos
WHERE r.en_vigor = TRUE
GROUP BY r.mes, r.linea_de_productos, lp.descripcion
ORDER BY r.mes ASC, r.linea_de_productos ASC;
"""

df_evolucion = pd.read_sql(query_evolucion_mensual_desc, con=engine)
df_evolucion["mes"] = pd.to_datetime(df_evolucion["mes"])

# Asegurar orden cronológico y calcular la media móvil de 6 meses por línea
df_evolucion = df_evolucion.sort_values(["linea_con_descripcion", "mes"])
df_evolucion["equs_ma_6m"] = df_evolucion.groupby("linea_con_descripcion")[
    "equs_totales"
].transform(lambda x: x.rolling(window=6, min_periods=1).mean())

# Generación del gráfico con media móvil
plt.figure(figsize=(14, 7))
sns.lineplot(
    data=df_evolucion,
    x="mes",
    y="equs_ma_6m",
    hue="linea_con_descripcion",
    marker="o",
    linewidth=2.5,
)

plt.title(
    "Evolución de Producción EQUS por Línea de Productos (Media Móvil 6 Meses)",
    fontsize=14,
    fontweight="bold",
    pad=15,
)
plt.xlabel("Mes", fontsize=11)
plt.ylabel("EQUS Totales (Promedio Móvil 6M)", fontsize=11)
plt.legend(title="Línea de Productos", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

Gran parte de la produccion bruta provienen de componentes de decorado.

## Autocorrelaciones mensuales

In [ ]:
# ==============================================================================
# A. Prueba de Autocorrelación (Ljung-Box) por LÍNEA DE PRODUCTOS
# ==============================================================================

# 1. Extraer o reutilizar el dataframe mensual por línea
query_lineas = """
SELECT
    r.mes,
    CONCAT(r.linea_de_productos, ' - ', COALESCE(lp.descripcion, 'Sin Desc.')) AS linea,
    SUM(r.valor * f.factor) AS equs_totales
FROM gold.v_reporte_qad_regla r
JOIN silver.factores_equ f
  ON f.version = r.version AND f.cuenta_stats = r.cuenta_stats
LEFT JOIN silver.linea_de_productos lp ON lp.linea_de_productos = r.linea_de_productos
WHERE r.en_vigor = TRUE
GROUP BY r.mes, r.linea_de_productos, lp.descripcion
ORDER BY r.mes ASC;
"""

df_lineas = pd.read_sql(query_lineas, con=engine)
df_lineas["mes"] = pd.to_datetime(df_lineas["mes"])

# 2. Aplicar test de Ljung-Box por línea (rezagos 1, 3 y 6 meses)
resultados_lineas = []
COMPARISON = 6
for linea, group in df_lineas.groupby("linea"):
    ts = group.sort_values("mes").set_index("mes")["equs_totales"].dropna()
    if len(ts) >= COMPARISON:  # Requiere suficiente longitud de serie
        lb_df = acorr_ljungbox(ts, lags=[1, 3, 6], return_df=True)
        lb_df["linea"] = linea
        lb_df["lag"] = lb_df.index
        resultados_lineas.append(lb_df)

df_autocorr_lineas = pd.concat(resultados_lineas, ignore_index=True)
df_autocorr_lineas["autocorrelacion_significativa"] = df_autocorr_lineas["lb_pvalue"] < ALPHA

print("--- Resultado Autocorrelación Ljung-Box (Líneas de Producto) ---")
display(
    df_autocorr_lineas[["linea", "lag", "lb_stat", "lb_pvalue", "autocorrelacion_significativa"]]
)


# ==============================================================================
# B. Prueba de Autocorrelación (Ljung-Box) por PRODUCTO (numero_articulo)
# ==============================================================================

query_productos = """
SELECT
    r.mes,
    r.numero_articulo,
    SUM(r.valor * f.factor) AS equs_totales
FROM gold.v_reporte_qad_regla r
JOIN silver.factores_equ f
  ON f.version = r.version AND f.cuenta_stats = r.cuenta_stats
WHERE r.en_vigor = TRUE
GROUP BY r.mes, r.numero_articulo
ORDER BY r.mes ASC;
"""

df_productos = pd.read_sql(query_productos, con=engine)
df_productos["mes"] = pd.to_datetime(df_productos["mes"])

# Filtrar productos con suficiente volumen/frecuencia
top_articulos = df_productos.groupby("numero_articulo")["equs_totales"].sum().nlargest(15).index

resultados_prod = []
COMPARISON2 = 7
for art in top_articulos:
    ts = (
        df_productos[df_productos["numero_articulo"] == art]
        .sort_values("mes")
        .set_index("mes")["equs_totales"]
    )
    if len(ts) >= COMPARISON2:
        lb_df = acorr_ljungbox(ts, lags=[1, 3, 6], return_df=True)
        lb_df["numero_articulo"] = art
        lb_df["lag"] = lb_df.index
        resultados_prod.append(lb_df)

df_autocorr_prod = pd.concat(resultados_prod, ignore_index=True)
df_autocorr_prod["autocorrelacion_significativa"] = df_autocorr_prod["lb_pvalue"] < ALPHA

print("\n--- Resultado Autocorrelación Ljung-Box (Top Productos) ---")
df_autocorr_prod[
    ["numero_articulo", "lag", "lb_stat", "lb_pvalue", "autocorrelacion_significativa"]
].head(10)

In [ ]:
df_lineas[df_lineas["linea"] == "1103 - Plomos Aleados"]["equs_totales"].var()

## Autocorrelaciones diarias

In [ ]:
# ==============================================================================
# A. Autocorrelación DIARIA (Ljung-Box) por LÍNEA DE PRODUCTOS
# ==============================================================================

query_lineas_diario = """
SELECT
    r.fecha_efectiva::date AS fecha,
    CONCAT(r.linea_de_productos, ' - ', COALESCE(lp.descripcion, 'Sin Desc.')) AS linea,
    SUM(r.valor * f.factor) AS equs_totales
FROM gold.v_reporte_qad_regla r
JOIN silver.factores_equ f
  ON f.version = r.version AND f.cuenta_stats = r.cuenta_stats
LEFT JOIN silver.linea_de_productos lp ON lp.linea_de_productos = r.linea_de_productos
WHERE r.en_vigor = TRUE
GROUP BY r.fecha_efectiva::date, r.linea_de_productos, lp.descripcion
ORDER BY fecha ASC;
"""

df_lineas_d = pd.read_sql(query_lineas_diario, con=engine)
df_lineas_d["fecha"] = pd.to_datetime(df_lineas_d["fecha"])

# Calendario continuo diario
rango_dias_lineas = pd.date_range(
    start=df_lineas_d["fecha"].min(), end=df_lineas_d["fecha"].max(), freq="D"
)

resultados_lineas_d = []
lags_diarios = [1, 7, 14, 30]

for linea, group in df_lineas_d.groupby("linea"):
    # Asegurar serie diaria continua sin duplicados
    ts = group.groupby("fecha")["equs_totales"].sum().reindex(rango_dias_lineas, fill_value=0.0)

    if len(ts) > max(lags_diarios):
        lb_df = acorr_ljungbox(ts, lags=lags_diarios, return_df=True)
        lb_df["linea"] = linea
        lb_df["lag"] = lb_df.index
        resultados_lineas_d.append(lb_df)

df_autocorr_lineas_d = pd.concat(resultados_lineas_d, ignore_index=True)
df_autocorr_lineas_d["autocorrelacion_significativa"] = df_autocorr_lineas_d["lb_pvalue"] < ALPHA

print("--- Resultado Autocorrelación Diaria (Líneas de Producto) ---")
display(
    df_autocorr_lineas_d[["linea", "lag", "lb_stat", "lb_pvalue", "autocorrelacion_significativa"]]
)


# ==============================================================================
# B. Autocorrelación DIARIA (Ljung-Box) por PRODUCTO (numero_articulo)
# ==============================================================================

query_productos_diario = """
SELECT
    r.fecha_efectiva::date AS fecha,
    r.numero_articulo,
    SUM(r.valor * f.factor) AS equs_totales
FROM gold.v_reporte_qad_regla r
JOIN silver.factores_equ f
  ON f.version = r.version AND f.cuenta_stats = r.cuenta_stats
WHERE r.en_vigor = TRUE
GROUP BY r.fecha_efectiva::date, r.numero_articulo
ORDER BY fecha ASC;
"""

df_prod_d = pd.read_sql(query_productos_diario, con=engine)
df_prod_d["fecha"] = pd.to_datetime(df_prod_d["fecha"])

# Top 15 artículos por volumen total diario
top_articulos_d = df_prod_d.groupby("numero_articulo")["equs_totales"].sum().nlargest(15).index

rango_dias_prod = pd.date_range(
    start=df_prod_d["fecha"].min(), end=df_prod_d["fecha"].max(), freq="D"
)
resultados_prod_d = []

for art in top_articulos_d:
    ts = (
        df_prod_d[df_prod_d["numero_articulo"] == art]
        .groupby("fecha")["equs_totales"]
        .sum()
        .reindex(rango_dias_prod, fill_value=0.0)
    )

    if len(ts) > max(lags_diarios):
        lb_df = acorr_ljungbox(ts, lags=lags_diarios, return_df=True)
        lb_df["numero_articulo"] = art
        lb_df["lag"] = lb_df.index
        resultados_prod_d.append(lb_df)

df_autocorr_prod_d = pd.concat(resultados_prod_d, ignore_index=True)
df_autocorr_prod_d["autocorrelacion_significativa"] = df_autocorr_prod_d["lb_pvalue"] < ALPHA

print("\n--- Resultado Autocorrelación Diaria (Top Productos) ---")
display(
    df_autocorr_prod_d[
        ["numero_articulo", "lag", "lb_stat", "lb_pvalue", "autocorrelacion_significativa"]
    ].head(12)
)

- Existe una fuerte dependencia de corto plazo. Lo que ocurre hoy en la planta o en despachos depende fuertemente de lo que pasó ayer (Lag 1), hace una semana (Lag 7) o hace un mes (Lag 30).

- Inercia y ritmo operativo: Los procesos de producción no arrancan ni frenan a cero de forma aleatoria de un día a otro. Hay turnos de trabajo, lotes de producción continuos, programas semanales de ensamble/formación y días de cierre/mantenimiento (como los fines de semana, lo que genera esa estacionalidad de 7 días).

- A nivel agregado mensual, los meses son independientes entre sí. El volumen total de EQUS producido en septiembre no determina ni condiciona directamente lo que se va a producir en octubre.

- Suavizado de la volatilidad diaria: Al sumar todos los días del mes para obtener el total mensual, la variación y el "ritmo" diario (los picos del miércoles o las caídas del domingo) se absorben dentro del total del mes.

# Estacionalidad Grafica

In [ ]:
# Configuración del estilo minimalista como el de la imagen
sns.set_theme(style="white")
plt.rcParams["font.sans-serif"] = "DejaVu Sans"

# ==============================================================================
# Consulta: EQUS Diarias por Día de la Semana y Línea de Producto
# ==============================================================================
query_estacionalidad_dia = """
SELECT
    r.fecha_efectiva::date AS fecha,
    -- Extraer el día de la semana (0=Lunes, 6=Domingo en PostgreSQL)
    EXTRACT(ISODOW FROM r.fecha_efectiva) AS num_dia,
    TO_CHAR(r.fecha_efectiva, 'Dy') AS nombre_dia,
    CONCAT(r.linea_de_productos, ' - ', COALESCE(lp.descripcion,
    'Sin Desc.')) AS linea_con_descripcion,
    SUM(r.valor * f.factor) AS equs_diarias
FROM gold.v_reporte_qad_regla r
JOIN silver.factores_equ f
  ON f.version = r.version AND f.cuenta_stats = r.cuenta_stats
LEFT JOIN silver.linea_de_productos lp
       ON lp.linea_de_productos = r.linea_de_productos
WHERE r.en_vigor = TRUE
GROUP BY r.fecha_efectiva::date, num_dia, nombre_dia, r.linea_de_productos, lp.descripcion
ORDER BY fecha ASC;
"""

df_estacionalidad = pd.read_sql(query_estacionalidad_dia, con=engine)

# Asegurar orden correcto de los días de la semana
dias_orden = ["Lun", "Mar", "Mié", "Jue", "Vie", "Sáb", "Dom"]
dias_map = {1: "Lun", 2: "Mar", 3: "Mié", 4: "Jue", 5: "Vie", 6: "Sáb", 7: "Dom"}

df_estacionalidad["dia_semana"] = df_estacionalidad["num_dia"].map(dias_map)
df_estacionalidad["dia_semana"] = pd.Categorical(
    df_estacionalidad["dia_semana"], categories=dias_orden, ordered=True
)

# ==============================================================================
# Generación del Gráfico (pointplot con barras de error)
# ==============================================================================
fig, ax = plt.subplots(figsize=(12, 6))

# Usamos pointplot para calcular el promedio por día y dibujar las barras de error
sns.pointplot(
    data=df_estacionalidad,
    x="dia_semana",
    y="equs_diarias",
    hue="linea_con_descripcion",
    errorbar=("ci", 95),  # Muestra el intervalo de confianza del 95% como en la imagen
    capsize=0.1,  # Ancho de los extremos de las barras de error
    markers="o",
    linestyles="-",
    ax=ax,
)

# Estilizado para imitar la imagen de referencia
ax.set_title(
    "Producción EQUS media por día de la semana", fontsize=14, fontweight="bold", loc="left", pad=15
)
ax.set_xlabel("")
ax.set_ylabel("EQUS medio por día", fontsize=11)

# Rejilla horizontal punteada tenue
ax.grid(axis="y", linestyle="--", alpha=0.5)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["left"].set_visible(False)
ax.spines["bottom"].set_linewidth(1.5)

plt.legend(title="Línea de Productos", bbox_to_anchor=(1.02, 1), loc="upper left", frameon=False)
plt.tight_layout()
plt.show()

In [ ]:
# Configuración del estilo minimalista
sns.set_theme(style="white")
plt.rcParams["font.sans-serif"] = "DejaVu Sans"

# ==============================================================================
# Consulta: EQUS Diarias por Mes del Año y Línea de Producto
# ==============================================================================
query_estacionalidad_mes = """
SELECT
    r.fecha_efectiva::date AS fecha,
    -- Extraer el número del mes (1=Enero, 12=Diciembre)
    EXTRACT(MONTH FROM r.fecha_efectiva) AS num_mes,
    CONCAT(r.linea_de_productos, ' - ', COALESCE(lp.descripcion,
    'Sin Desc.')) AS linea_con_descripcion,
    SUM(r.valor * f.factor) AS equs_diarias
FROM gold.v_reporte_qad_regla r
JOIN silver.factores_equ f
  ON f.version = r.version AND f.cuenta_stats = r.cuenta_stats
LEFT JOIN silver.linea_de_productos lp
       ON lp.linea_de_productos = r.linea_de_productos
WHERE r.en_vigor = TRUE
GROUP BY r.fecha_efectiva::date, num_mes, r.linea_de_productos, lp.descripcion
ORDER BY fecha ASC;
"""

df_estacionalidad_mes = pd.read_sql(query_estacionalidad_mes, con=engine)

# Mapeo y ordenamiento categórico de los meses del año (Ene -> Dic)
meses_orden = ["Ene", "Feb", "Mar", "Abr", "May", "Jun", "Jul", "Ago", "Sep", "Oct", "Nov", "Dic"]
meses_map = {
    1: "Ene",
    2: "Feb",
    3: "Mar",
    4: "Abr",
    5: "May",
    6: "Jun",
    7: "Jul",
    8: "Ago",
    9: "Sep",
    10: "Oct",
    11: "Nov",
    12: "Dic",
}

df_estacionalidad_mes["nombre_mes"] = df_estacionalidad_mes["num_mes"].map(meses_map)
df_estacionalidad_mes["nombre_mes"] = pd.Categorical(
    df_estacionalidad_mes["nombre_mes"], categories=meses_orden, ordered=True
)

# ==============================================================================
# Generación del Gráfico (pointplot por mes)
# ==============================================================================
fig, ax = plt.subplots(figsize=(13, 6))

sns.pointplot(
    data=df_estacionalidad_mes,
    x="nombre_mes",
    y="equs_diarias",
    hue="linea_con_descripcion",
    errorbar=("ci", 95),  # Intervalo de confianza del 95% para la media del mes
    capsize=0.1,
    markers="o",
    linestyles="-",
    ax=ax,
)

# Estilizado
ax.set_title(
    "Producción EQUS media diaria por mes del año",
    fontsize=14,
    fontweight="bold",
    loc="left",
    pad=15,
)
ax.set_xlabel("")
ax.set_ylabel("EQUS medio diario", fontsize=11)

# Rejilla horizontal tenue y eliminación de bordes sobrantes
ax.grid(axis="y", linestyle="--", alpha=0.5)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["left"].set_visible(False)
ax.spines["bottom"].set_linewidth(1.5)

plt.legend(title="Línea de Productos", bbox_to_anchor=(1.02, 1), loc="upper left", frameon=False)
plt.tight_layout()
plt.show()

In [ ]:
# Configuración del estilo minimalista
sns.set_theme(style="white")
plt.rcParams["font.sans-serif"] = "DejaVu Sans"

# ==============================================================================
# Consulta: EQUS Diarias por Mes Fiscal y Año Fiscal (FY)
# ==============================================================================
query_estacionalidad_fy = """
SELECT
    r.fecha_efectiva::date AS fecha,
    -- Extraer mes y año
    EXTRACT(MONTH FROM r.fecha_efectiva) AS num_mes,
    EXTRACT(YEAR FROM r.fecha_efectiva) AS num_anio,
    -- Definir Año Fiscal (FY): Si el mes es >= 10
    -- (Oct, Nov, Dic), pertenece al FY del año siguiente
    CASE
        WHEN EXTRACT(MONTH FROM r.fecha_efectiva) >= 10
        THEN CONCAT('FY', (EXTRACT(YEAR FROM r.fecha_efectiva) + 1)::text)
        ELSE CONCAT('FY', EXTRACT(YEAR FROM r.fecha_efectiva)::text)
    END AS fiscal_year,
    SUM(r.valor * f.factor) AS equs_diarias
FROM gold.v_reporte_qad_regla r
JOIN silver.factores_equ f
  ON f.version = r.version AND f.cuenta_stats = r.cuenta_stats
WHERE r.en_vigor = TRUE
GROUP BY r.fecha_efectiva::date, num_mes, num_anio, fiscal_year
ORDER BY fecha ASC;
"""

df_fy = pd.read_sql(query_estacionalidad_fy, con=engine)

# Mapeo y ordenamiento del año fiscal (Octubre a Septiembre)
meses_fiscales_orden = [
    "Oct",
    "Nov",
    "Dic",
    "Ene",
    "Feb",
    "Mar",
    "Abr",
    "May",
    "Jun",
    "Jul",
    "Ago",
    "Sep",
]
meses_map = {
    10: "Oct",
    11: "Nov",
    12: "Dic",
    1: "Ene",
    2: "Feb",
    3: "Mar",
    4: "Abr",
    5: "May",
    6: "Jun",
    7: "Jul",
    8: "Ago",
    9: "Sep",
}

df_fy["mes_fiscal"] = df_fy["num_mes"].map(meses_map)
df_fy["mes_fiscal"] = pd.Categorical(
    df_fy["mes_fiscal"], categories=meses_fiscales_orden, ordered=True
)

# ==============================================================================
# Generación del Gráfico (pointplot por Mes Fiscal comparando FYs)
# ==============================================================================
fig, ax = plt.subplots(figsize=(13, 6))

sns.pointplot(
    data=df_fy,
    x="mes_fiscal",
    y="equs_diarias",
    hue="fiscal_year",
    errorbar=("ci", 95),  # IC del 95% para la media diaria de cada mes en ese FY
    capsize=0.1,
    markers="o",
    linestyles="-",
    palette="tab10",
    ax=ax,
)

# Estilizado para igualar la imagen de referencia
ax.set_title(
    "Producción EQUS media por mes fiscal (Oct → Sep)",
    fontsize=14,
    fontweight="bold",
    loc="left",
    pad=15,
)
ax.set_xlabel("")
ax.set_ylabel("EQUS medio diario", fontsize=11)

# Rejilla horizontal tenue y limpieza de ejes
ax.grid(axis="y", linestyle="--", alpha=0.5)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["left"].set_visible(False)
ax.spines["bottom"].set_linewidth(1.5)

plt.legend(title="Año Fiscal", bbox_to_anchor=(1.02, 1), loc="upper left", frameon=False)
plt.tight_layout()
plt.show()

In [ ]:
# Configuración del estilo visual
sns.set_theme(style="white")
plt.rcParams["font.sans-serif"] = "DejaVu Sans"

# ==============================================================================
# Consulta: EQUS Diarias por Mes Fiscal (Oct -> Sep) y Línea de Producto
# ==============================================================================
query_estacionalidad_lineas = """
SELECT
    r.fecha_efectiva::date AS fecha,
    -- Extraer mes y año
    EXTRACT(MONTH FROM r.fecha_efectiva) AS num_mes,
    -- Definir Año Fiscal (FY): Si el mes es >= 10 (Oct, Nov, Dic),
    -- pertenece al FY del año siguiente
    CASE
        WHEN EXTRACT(MONTH FROM r.fecha_efectiva) >= 10
        THEN CONCAT('FY', (EXTRACT(YEAR FROM r.fecha_efectiva) + 1)::text)
        ELSE CONCAT('FY', EXTRACT(YEAR FROM r.fecha_efectiva)::text)
    END AS fiscal_year,
    -- Etiqueta de la línea de producto
    CONCAT(r.linea_de_productos, ' - ',
    COALESCE(lp.descripcion, 'Sin Desc.')) AS linea_con_descripcion,
    SUM(r.valor * f.factor) AS equs_diarias
FROM gold.v_reporte_qad_regla r
JOIN silver.factores_equ f
  ON f.version = r.version AND f.cuenta_stats = r.cuenta_stats
LEFT JOIN silver.linea_de_productos lp
       ON lp.linea_de_productos = r.linea_de_productos
WHERE r.en_vigor = TRUE
GROUP BY r.fecha_efectiva::date, num_mes, fiscal_year, r.linea_de_productos, lp.descripcion
ORDER BY fecha ASC;
"""

df_estacionalidad_lineas = pd.read_sql(query_estacionalidad_lineas, con=engine)

# Mapeo y ordenamiento del año fiscal (Octubre a Septiembre)
meses_fiscales_orden = [
    "Oct",
    "Nov",
    "Dic",
    "Ene",
    "Feb",
    "Mar",
    "Abr",
    "May",
    "Jun",
    "Jul",
    "Ago",
    "Sep",
]
meses_map = {
    10: "Oct",
    11: "Nov",
    12: "Dic",
    1: "Ene",
    2: "Feb",
    3: "Mar",
    4: "Abr",
    5: "May",
    6: "Jun",
    7: "Jul",
    8: "Ago",
    9: "Sep",
}

df_estacionalidad_lineas["mes_fiscal"] = df_estacionalidad_lineas["num_mes"].map(meses_map)
df_estacionalidad_lineas["mes_fiscal"] = pd.Categorical(
    df_estacionalidad_lineas["mes_fiscal"], categories=meses_fiscales_orden, ordered=True
)

# ==============================================================================
# Generación del Gráfico (pointplot por Mes Fiscal por Línea de Producto)
# ==============================================================================
fig, ax = plt.subplots(figsize=(13, 6))

sns.pointplot(
    data=df_estacionalidad_lineas,
    x="mes_fiscal",
    y="equs_diarias",
    hue="linea_con_descripcion",
    errorbar=("ci", 95),  # IC del 95% para el promedio diario del mes
    capsize=0.1,
    markers="o",
    linestyles="-",
    ax=ax,
)

# Estilizado
ax.set_title(
    "Producción EQUS media por mes fiscal (Oct → Sep) por línea",
    fontsize=14,
    fontweight="bold",
    loc="left",
    pad=15,
)
ax.set_xlabel("")
ax.set_ylabel("EQUS medio por día", fontsize=11)

# Rejilla horizontal tenue y eliminación de bordes sobrantes
ax.grid(axis="y", linestyle="--", alpha=0.5)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["left"].set_visible(False)
ax.spines["bottom"].set_linewidth(1.5)

plt.legend(title="Línea de Productos", bbox_to_anchor=(1.02, 1), loc="upper left", frameon=False)
plt.tight_layout()
plt.show()

In [ ]:
# ==============================================================================
# Consulta SQL: Granularidad mínima (segundo) agrupada por producto
# ==============================================================================
query_granularidad_minima = """
WITH transacciones_segundo AS (
    -- 1. Transacciones de producción (QAD) a nivel de segundo
    SELECT
        date_trunc('second', r.fecha_efectiva::timestamp) AS timestamp_segundo,
        r.fecha_efectiva::date AS fecha,
        r.mes,
        r.version,
        r.proceso,
        r.cuenta_stats,
        r.numero_articulo,
        CONCAT(r.numero_articulo,
        ' - ',
        COALESCE(i.descripcion, 'Sin Descripción'))
        AS producto_con_descripcion,
        r.linea_de_productos,
        r.linea_produccion,
        r.almac,
        'PRODUCCION' AS tipo_origen,
        r.cantidad AS unidades_fisicas,
        (r.valor * f.factor) AS equs
    FROM gold.v_reporte_qad_regla r
    JOIN silver.factores_equ f
      ON f.version = r.version AND f.cuenta_stats = r.cuenta_stats
    LEFT JOIN silver.item_master i
           ON i.numero_articulo = r.numero_articulo
    WHERE r.en_vigor = TRUE

    UNION ALL

    -- 2. Transacciones de facturación (Unit Ship) a nivel de segundo
    SELECT
        date_trunc('second', s.fecha_efectiva::timestamp) AS timestamp_segundo,
        s.fecha_efectiva::date AS fecha,
        s.mes,
        s.version,
        s.proceso,
        s.cuenta_stats,
        s.numero_articulo,
        CONCAT(
        s.numero_articulo,
        ' - ',
        COALESCE(i.descripcion, 'Sin Descripción'))
        AS producto_con_descripcion,
        s.linea_de_productos,
        NULL AS linea_produccion,
        NULL AS almac,
        'VENTA' AS tipo_origen,
        s.cantidad AS unidades_fisicas,
        (s.valor * f.factor) AS equs
    FROM gold.v_unit_ship_regla s
    JOIN silver.factores_equ f
      ON f.version = s.version AND f.cuenta_stats = s.cuenta_stats
    LEFT JOIN silver.item_master i
           ON i.numero_articulo = s.numero_articulo
    WHERE s.en_vigor = TRUE
)
SELECT
    t.timestamp_segundo,
    t.fecha,
    t.mes,
    t.version,
    t.proceso,
    t.cuenta_stats,
    t.numero_articulo,
    t.producto_con_descripcion,
    t.linea_de_productos,
    t.linea_produccion,
    t.almac,
    t.tipo_origen,
    SUM(t.unidades_fisicas) AS unidades_totales,
    SUM(t.equs) AS equs_totales,
    COUNT(*) AS cantidad_transacciones
FROM transacciones_segundo t
GROUP BY
    t.timestamp_segundo,
    t.fecha,
    t.mes,
    t.version,
    t.proceso,
    t.cuenta_stats,
    t.numero_articulo,
    t.producto_con_descripcion,
    t.linea_de_productos,
    t.linea_produccion,
    t.almac,
    t.tipo_origen
ORDER BY
    t.timestamp_segundo ASC,
    t.numero_articulo ASC;
"""

# Cargar en el DataFrame
df_minima_granularidad = pd.read_sql(query_granularidad_minima, con=engine)
df_minima_granularidad["timestamp_segundo"] = pd.to_datetime(
    df_minima_granularidad["timestamp_segundo"]
)

# Vista previa del DataFrame resultante
print(df_minima_granularidad.info())
display(df_minima_granularidad.head(10))

In [ ]:
df_minima_granularidad["proceso"].unique()

In [ ]:
df_minima_granularidad.shape

# Ahora cruzar df_detalle_diario con los SCADA

In [ ]:
RUTA = Path("../data/SCADA-Base de datos F&A.xlsx")
raw = pd.read_excel(RUTA, sheet_name=HOJA)

In [ ]:
raw

In [ ]:
dias_unicos = pd.to_datetime(raw["t_stamp"]).dt.date.nunique()
print(f"El conjunto 'raw' contiene {dias_unicos} días únicos.")

In [ ]:
# ==============================================================================
# 1. Filtrar df_minima_granularidad (ENSAMBLE y FORMACION) y agregar por Línea
# ==============================================================================
procesos_objetivo = ["ENSAMBLE", "FORMACION"]

# Filtro directo sobre el DataFrame definitivo
df_equs_filtrado = df_minima_granularidad[
    df_minima_granularidad["proceso"].isin(procesos_objetivo)
].copy()

# Asegurar formato fecha (date) y etiqueta con descripción
df_equs_filtrado["fecha"] = pd.to_datetime(df_equs_filtrado["fecha"]).dt.date
if "linea_con_descripcion" not in df_equs_filtrado.columns:
    df_equs_filtrado["linea_con_descripcion"] = df_equs_filtrado["linea_de_productos"].astype(str)

# A. Agregación por Fecha y Proceso (ENSAMBLE / FORMACION)
df_proceso_diario = df_equs_filtrado.groupby(["fecha", "proceso"], as_index=False).agg(
    unidades_totales=("unidades_totales", "sum"), equs_totales=("equs_totales", "sum")
)

df_proceso_pivot = df_proceso_diario.pivot(
    index="fecha", columns="proceso", values=["unidades_totales", "equs_totales"]
)
df_proceso_pivot.columns = [
    f"{metrica}_{proceso.lower()}" for metrica, proceso in df_proceso_pivot.columns
]
df_proceso_pivot = df_proceso_pivot.reset_index()


# B. Agregación por Fecha y Línea de Producto
df_linea_diario = df_equs_filtrado.groupby(["fecha", "linea_con_descripcion"], as_index=False).agg(
    unidades_totales=("unidades_totales", "sum"), equs_totales=("equs_totales", "sum")
)

df_linea_pivot = df_linea_diario.pivot(
    index="fecha", columns="linea_con_descripcion", values=["unidades_totales", "equs_totales"]
)
df_linea_pivot.columns = [f"{metrica}_linea_{linea}" for metrica, linea in df_linea_pivot.columns]
df_linea_pivot = df_linea_pivot.reset_index()


# C. Consolidar la parte de EQUS (Proceso + Líneas)
df_equs_consolidado = pd.merge(df_proceso_pivot, df_linea_pivot, on="fecha", how="outer").fillna(0)


# ==============================================================================
# 2. Procesar SCADA (raw): Promedio diario y Desviación Estándar (std)
# ==============================================================================
df_scada = raw.copy()
df_scada["t_stamp"] = pd.to_datetime(df_scada["t_stamp"])
df_scada["fecha"] = df_scada["t_stamp"].dt.date

cols_scada = [c for c in df_scada.columns if c not in ["t_stamp", "fecha"]]

df_scada_stats = df_scada.groupby("fecha")[cols_scada].agg(["mean", "std"])

df_scada_stats.columns = [f"{sensor}_{stat}" for sensor, stat in df_scada_stats.columns]
df_scada_stats = df_scada_stats.reset_index().fillna(0)


# ==============================================================================
# 3. Cruce final (Merge) por fecha
# ==============================================================================
df_modelamiento = (
    pd.merge(df_equs_consolidado, df_scada_stats, on="fecha", how="inner")
    .sort_values("fecha")
    .reset_index(drop=True)
)

print(f"Dimensiones del dataset definitivo: {df_modelamiento.shape}")
display(df_modelamiento)

In [ ]:
fechas_scada = set(pd.to_datetime(raw["t_stamp"]).dt.date)
fechas_equs = set(df_equs_consolidado["fecha"])

fechas_faltantes = sorted(fechas_scada - fechas_equs)

print(f"Días en SCADA sin registro de producción ({len(fechas_faltantes)} días):")
print(fechas_faltantes)

In [ ]:
# 1. Extraer los conjuntos únicos de fechas de cada fuente
fechas_scada = set(df_scada_stats["fecha"])
fechas_equs = set(df_equs_consolidado["fecha"])

# 2. Identificar las fechas que están en SCADA pero no en el reporte de producción
fechas_solo_scada = fechas_scada - fechas_equs

# 3. Construir la tabla de control con las métricas del SCADA en esas fechas sin producción
df_control_faltantes = (
    df_scada_stats[df_scada_stats["fecha"].isin(fechas_solo_scada)]
    .sort_values("fecha")
    .reset_index(drop=True)
)

# Agregar columnas explícitas de motivo y estado de producción
df_control_faltantes.insert(1, "estado_produccion", "Sin registro de producción (0 EQUS)")
df_control_faltantes.insert(
    2, "motivo_observado", "Día sin transacciones QAD (Posible parada de planta / Fin de semana)"
)

print(f"Total de días en control sin producción: {len(df_control_faltantes)}")
display(df_control_faltantes.head(10))

In [ ]:
# ==============================================================================
# 1. Crear el Dataset Comparativo (Con EQUS vs Sin EQUS)
# ==============================================================================
df_comparativo = df_scada_stats.copy()

# Etiquetar cada fecha según si tuvo registro de producción
fechas_con_equs = set(df_equs_consolidado["fecha"])
df_comparativo["grupo_produccion"] = df_comparativo["fecha"].apply(
    lambda f: "Con EQUS" if f in fechas_con_equs else "Sin EQUS"
)

# Identificar las columnas de promedios (_mean) de los rectificadores/sensores
cols_mean = [c for c in df_comparativo.columns if c.endswith("_mean")]

# ==============================================================================
# 2. Resumen Estadístico (Medias por Grupo)
# ==============================================================================
resumen_medias = (
    df_comparativo.groupby("grupo_produccion")[cols_mean]
    .mean()
    .T.rename(columns={"Con EQUS": "Media (Con EQUS)", "Sin EQUS": "Media (Sin EQUS)"})
)

# Calcular diferencia absoluta y relativa (%)
resumen_medias["Diferencia_Absoluta"] = (
    resumen_medias["Media (Con EQUS)"] - resumen_medias["Media (Sin EQUS)"]
)
resumen_medias["Diferencia_Pct (%)"] = (
    resumen_medias["Diferencia_Absoluta"] / resumen_medias["Media (Sin EQUS)"].replace(0, np.nan)
) * 100

# ==============================================================================
# 3. Prueba t de Student / Welch para validar significancia estadística
# ==============================================================================
p_values = []
t_stats = []

grupo_con = df_comparativo[df_comparativo["grupo_produccion"] == "Con EQUS"]
grupo_sin = df_comparativo[df_comparativo["grupo_produccion"] == "Sin EQUS"]

for col in cols_mean:
    # Welch's t-test (no asume varianzas iguales)
    stat, pval = stats.ttest_ind(grupo_con[col], grupo_sin[col], equal_var=False, nan_policy="omit")
    t_stats.append(stat)
    p_values.append(pval)

resumen_medias["t_stat"] = t_stats
resumen_medias["p_value"] = p_values
resumen_medias["diferencia_significativa"] = resumen_medias["p_value"] < ALPHA

print("--- Comparativa de Medias SCADA: Días Con EQUS vs Días Sin EQUS ---")
display(resumen_medias.round(4))

A pesar de no registrar produccion los valores medios de consumo son similares a los dias que si registran equivalentes.

In [ ]:
# Reestructurar DataFrame para plotting fácil con Seaborn
df_melted = df_comparativo.melt(
    id_vars=["fecha", "grupo_produccion"],
    value_vars=cols_mean,
    var_name="sensor",
    value_name="promedio_diario",
)

# Limpiar nombres de sensores para la etiqueta
df_melted["sensor"] = df_melted["sensor"].str.replace("_mean", "")

plt.figure(figsize=(12, 6))
sns.barplot(
    data=df_melted,
    x="sensor",
    y="promedio_diario",
    hue="grupo_produccion",
    palette="Set2",
    errorbar=("ci", 95),
    capsize=0.1,
)

plt.title(
    "Comparación de Medias SCADA por Rectificador (Días Con EQUS vs Sin EQUS)",
    fontsize=13,
    fontweight="bold",
    pad=15,
)
plt.xlabel("Sensor / Rectificador", fontsize=11)
plt.ylabel("Valor Promedio Diario", fontsize=11)
plt.xticks(rotation=45)
plt.legend(title="Grupo", frameon=True)
plt.tight_layout()
plt.show()